In [ ]:
import sys
import os

from bvpy.domains import CustomDomain, CustomDomainGmsh
from bvpy.utils.pre_processing import HeterogeneousParameter
from bvpy.utils.io import save
import pyvista as pv
from bvpy.utils.visu_pyvista import visualize
from bvpy.vforms import LinearElasticForm, HyperElasticForm, StVenantKirchoffPotential
from bvpy.boundary_conditions import dirichlet, NormalDirichlet, NormalNeumann, neumann, Boundary
from bvpy.utils.visu import plot
import fenics as fe
import numpy as np
import sys
import meshio as mio
from bvpy import BVP
from subprocess import call, DEVNULL
import pandas as pd

from bvpy.gbvp import GBVP
from bvpy.vforms.plasticity import MorphoElasticForm, ConstantGrowth, StrainDependentGrowth, TimeDependentGrowth, HeterogeneousGrowth
from bvpy.utils.post_processing import SolutionExplorer


def xdmf_save(path, solution, vform, Fg, timestep):   
    """
    Save the displacement, strain, and stress fields to an XDMF file for a specific time step.

    Args:
        path (str): The file path to save the XDMF file.
        solution: The displacement vector solution.
        vform: Variational form object with strain and stress computation methods.
        Fg: Growth tensor for the current time step.
        time_step (float): The current simulation time step.
    """
    solution.rename("Displacement Vector", "")
    strain_e = vform.get_strain(solution, method='elastic', F_g=Fg)
    strain_t = vform.get_strain(solution, method='total', F_g=Fg)
    strain_e.rename("Elastic Strain", "")
    strain_t.rename("Total Strain", "")
    stress_e = vform.get_stress(solution, method='elastic', F_g=Fg)
    stress_t = vform.get_stress(solution, method='total', F_g=Fg)
    stress_e.rename("Elastic Stress", "")
    stress_t.rename("Total Stress", "")

    xdmf_file = fe.XDMFFile(fe.MPI.comm_world, path)
    xdmf_file.parameters["flush_output"] = True
    xdmf_file.parameters["functions_share_mesh"] = True
    xdmf_file.parameters["rewrite_function_mesh"] = False    
    xdmf_file.write(solution, timestep)
    xdmf_file.write(strain_e, timestep)
    xdmf_file.write(strain_t, timestep)
    xdmf_file.write(stress_e, timestep)
    xdmf_file.write(stress_t, timestep)
    
# Func for change mesh size
def generate_mesh(mesh_file, scale):
    print("gmsh "+ mesh_file+ " -2"+ " -clscale "+ f'{scale}')
    call(["gmsh", mesh_file, "-2", "-clscale", f'{scale}'], stdout=DEVNULL, stderr=DEVNULL)

In [ ]:
mesh_file = '../../data/hook_wall.geo'
scale = 1.3
generate_mesh(mesh_file, scale)
mesh_path = '../../data/hook_wall.msh'
cd = CustomDomainGmsh(fname = mesh_path)

In [ ]:
pl = visualize(cd, visu_type='domain', cmap='viridis')
pl.view_xy()

In [ ]:
ymin = cd.mesh.coordinates()[:, 1].min()
hook_base = Boundary(f'near(y, {ymin}, 1)')

bc = dirichlet([0.,0.0, 0.0], boundary= hook_base)

In [ ]:
df = pd.read_csv('../../data/IDlabel_young.csv')
young_values_by_labels = {row.ID_label: row.Young_value for _, row in df.iterrows()}
heterogeneous_young = HeterogeneousParameter(cd.cdata, young_values_by_labels)
elastic_potential = StVenantKirchoffPotential(young=heterogeneous_young, poisson=0.4)
heterogeneous_Hyperelastic_response = HyperElasticForm(potential_energy=elastic_potential, source=[0., 0., 0.],
                                                       plane_stress=True)


In [ ]:
growth_df = pd.read_csv("../../data/growth_data.csv")  # growth data

In [ ]:
plastic_vform = MorphoElasticForm(potential_energy=elastic_potential, F_g=np.identity(3), source = [0.,0.,0.])
# 
growth_classes = {}
for _, row in growth_df.iterrows():
    id_label = int(row['ID_label'])
    x_growth = row['sig_xx']
    y_growth = row['sig_yy']
    xy_growth = row['sig_xy']

    # Create a ConstantGrowth object for each ID
    growth_classes[id_label] = ConstantGrowth(dg=np.array([[x_growth/160, xy_growth/160, 0.], [xy_growth/160, y_growth/160, 0], [0.,0.,0.]])
    )

    

In [ ]:
# Define the growth scheme

#growth_scheme = StrainDependentGrowth(plastic_vform=plastic_vform, extensibility=2., target_strain=0.05)
growth_scheme = HeterogeneousGrowth(cdata = cd.cdata, growth_classes=growth_classes)


In [ ]:
# - Define problem
growth_problem = GBVP(domain=cd, vform=plastic_vform, bc=bc, growth_scheme=growth_scheme)

In [ ]:
growth_problem.info()

In [ ]:
growth_problem.run(tmin=0, tmax=8, dt=0.1, store_steps=True, 
                   linear_solver='mumps', krylov_solver={'absolute_tolerance':1e-14}, 
                   relative_tolerance = 1e-6, absolute_tolerance = 1e-8,
                   preconditioner = 'none',
                   maximum_iterations = 500, line_search='bt')

In [ ]:
time_points = list(growth_problem.solution_steps.keys())
print(time_points)

In [ ]:
int_length = len(str(len(time_points)))+1
for ix, t in enumerate(time_points):
    li = int(int_length-len(str(ix)))
    formatted_ix = ''.join(['0'] *  li)
    solution = growth_problem.solution_steps[t]
    Fg = growth_problem.growth_steps[t]
    save_path = f'./data/sim_{formatted_ix}{ix}.xdmf'
    xdmf_save(save_path, solution, vform = plastic_vform, Fg= Fg, timestep = t)

    